# 4. SAM Configuration
Verifica che il modello SAM funzioni e scarica eventuali file base.

In [4]:
import os, torch, subprocess
from pathlib import Path
subprocess.run(['pip', 'install', '-q', 'ultralytics==8.3.86', 'numpy<2'], check=True)
from ultralytics import SAM

In [5]:
BASE = Path('/teamspace/studios/this_studio/project-sbs')
WEIGHTS_SAM = BASE / 'model_weights' / 'sam'
WEIGHTS_SAM.mkdir(parents=True, exist_ok=True)
SAM_MODEL_PATH = WEIGHTS_SAM / 'sam2.1_l.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
print('Caricamento modello SAM...')
try:
    sam_model = SAM('sam2.1_l.pt')
    if Path('sam2.1_l.pt').exists():
        os.rename('sam2.1_l.pt', SAM_MODEL_PATH)
        sam_model = SAM(str(SAM_MODEL_PATH))
    print(f'✅ SAM 2.1 Large caricato su {DEVICE}')
except Exception as e:
    print(f'❌ Errore caricamento SAM: {e}')

Caricamento modello SAM...


100%|██████████| 428M/428M [00:01<00:00, 239MB/s]  


✅ SAM 2.1 Large caricato su cuda


In [ ]:
#CONFRONTO SAM E DATASET 2
import json, cv2, numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

VAL_SAM_DIR = BASE / 'datasets' / 'dataset_sam' / 'val'

if not VAL_SAM_DIR.exists():
    print("❌ Cartella validation di SAM non trovata.")
else:
    print("📊 Inizio valutazione SAM...")
    json_files = list(VAL_SAM_DIR.glob('*.json'))
    ious = []
    
    for jpath in tqdm(json_files, desc="Calcolo IoU"):
        idx = jpath.stem
        img_p = VAL_SAM_DIR / f"frame{idx}.jpg"
        if not img_p.exists(): img_p = VAL_SAM_DIR / f"frame{idx}.png"
        if not img_p.exists(): continue
            
        data = json.loads(jpath.read_text())
        regions = data.get('regions', [])
        if not regions: continue
            
        img = cv2.imread(str(img_p))
        if img is None: continue
        
        # 1. Maschera Ground Truth (Perfetta)
        gt_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        all_x = regions[0]['all_points_x']
        all_y = regions[0]['all_points_y']
        pts = np.column_stack((all_x, all_y)).astype(np.int32)
        cv2.fillPoly(gt_mask, [pts], 1)
        
        # Ricaviamo il Bounding Box perfetto dal GS per aiutare SAM 
        x, y, w, h = cv2.boundingRect(pts)
        gt_box = [x, y, x+w, y+h]
        
        # 2. Maschera Predetta da SAM
        sam_res = sam_model(img, bboxes=[gt_box], verbose=False)
        if not sam_res[0].masks: continue
        
        m_data = sam_res[0].masks.data.cpu().numpy()
        pred_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for m in m_data:
            if m.shape != img.shape[:2]:
                m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            pred_mask = np.maximum(pred_mask, (m > 0.5).astype(np.uint8))
            
        # 3. Metrica IoU
        intersection = np.logical_and(gt_mask, pred_mask).sum()
        union = np.logical_or(gt_mask, pred_mask).sum()
        
        if union > 0:
            ious.append(intersection / union)

    if ious:
        mean_iou = sum(ious) / len(ious)
        print(f"\n✅ Valutazione completata su {len(ious)} immagini.")
        print(f"⭐ Mean Intersection over Union (mIoU): {mean_iou:.4f} ({mean_iou*100:.2f}%)")
    else:
        print("Test fallito: nessun dato valido.")

# ── Confronto Visivo: SAM vs Ground Truth ──

In [ ]:
import random
if 'json_files' in locals() and json_files:
    jpath = random.choice(json_files)
    idx = jpath.stem
    img_p = VAL_SAM_DIR / f"frame{idx}.jpg"
    if not img_p.exists(): img_p = VAL_SAM_DIR / f"frame{idx}.png"
    
    if img_p.exists():
        data = json.loads(jpath.read_text())
        regions = data.get('regions', [])
        img = cv2.imread(str(img_p))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if regions:
            # GT
            gt_mask = np.zeros(img.shape[:2], dtype=np.uint8)
            pts = np.column_stack((regions[0]['all_points_x'], regions[0]['all_points_y'])).astype(np.int32)
            cv2.fillPoly(gt_mask, [pts], 1)
            x, y, w, h = cv2.boundingRect(pts)
            
            # SAM
            sam_res = sam_model(img_rgb, bboxes=[[x, y, x+w, y+h]], verbose=False)
            if hasattr(sam_res[0], 'names'): sam_res[0].names[0] = 'SAM'
            
            fig, axes = plt.subplots(1, 2, figsize=(18, 6))
            
            # Overlay GT (Verde)
            gt_overlay = img_rgb.copy()
            gt_overlay[gt_mask == 1] = [0, 255, 0]
            axes[0].imshow(cv2.addWeighted(img_rgb, 0.4, gt_overlay, 0.6, 0))
            axes[0].set_title("Ground Truth Perfetto (Maschera del Dataset)")
            axes[0].axis('off')
            
            # Overlay SAM
            axes[1].imshow(sam_res[0].plot(boxes=False))
            axes[1].set_title("Predizione SAM (Zero-Shot)")
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()